# Sistema multiagente de soporte con RAG y Langfuse

Un **Agente Orquestador** clasifica cada consulta (HR, Tech, Finance o fuera de dominio) y la delega con enrutamiento condicional a un **Agente RAG** especializado. Todo el flujo está trazado en **Langfuse** y un **Agente Evaluador** puntúa cada respuesta.

El código vive en `src/`; este notebook lo importa y lo explica paso a paso. Ejecuta las celdas **en orden**.

## 1. Setup e imports

Cargamos la configuración desde `.env`. Mostramos solo si cada clave está presente, nunca su valor.

In [1]:
from src.config import cargar_config, crear_llm, DEPARTAMENTOS
from src.indexacion import dividir_documentos, indexar_todo
from src.agents.orchestrator import crear_orquestador
from src.agents.hr_agent import crear_hr_agent
from src.agents.tech_agent import crear_tech_agent
from src.agents.finance_agent import crear_finance_agent
from src.multi_agent_system import crear_sistema, responder, medir_test_queries
from src.observabilidad import langfuse_configurado

config = cargar_config()
print('Modelo de chat:', config['modelo_chat'])
print('Modelo de embeddings:', config['modelo_embeddings'])
print('Clave de OpenAI presente:', bool(config['openai_api_key']))
print('Langfuse configurado:', langfuse_configurado(config))

Modelo de chat: gpt-6-luna
Modelo de embeddings: text-embedding-3-small
Clave de OpenAI presente: True
Langfuse configurado: True


## 2. Carga de documentos y vector stores

Cada departamento tiene su carpeta en `data/` y su propia colección en Chroma. Los documentos se dividen primero por títulos y luego en chunks de 500 caracteres.

In [2]:
for departamento in DEPARTAMENTOS:
    chunks = dividir_documentos(departamento)
    print(f'{departamento}: {len(chunks)} chunks')

ejemplo = dividir_documentos('finance')[0]
print('\nEjemplo de chunk:', ejemplo.metadata)
print(ejemplo.page_content[:300])

hr: 86 chunks
tech: 74 chunks
finance: 70 chunks

Ejemplo de chunk: {'fuente': 'cierre_de_mes.md', 'seccion': 'Cierre contable de mes — NubeGestor', 'departamento': 'finance'}
# Cierre contable de mes — NubeGestor  
Este documento del equipo de Finanzas (Finance) explica cómo funciona el cierre contable mensual de NubeGestor, qué plazos deben cumplir las demás áreas y qué reportes se generan. Un cierre ordenado permite tener información financiera confiable para la direcc


In [3]:
# Crea los embeddings solo si las colecciones están vacías (la primera vez).
conteos = indexar_todo(config)
conteos

[indexación] hr: 86 chunks
[indexación] tech: 74 chunks
[indexación] finance: 70 chunks


{'hr': 86, 'tech': 74, 'finance': 70}

## 3. Definición de agentes

Los tres agentes comparten la misma fábrica (`crear_agente_rag`): retriever de su colección + prompt de rol + modelo. Probamos uno directamente, sin orquestador.

In [4]:
agentes = {
    'hr': crear_hr_agent(config),
    'tech': crear_tech_agent(config),
    'finance': crear_finance_agent(config),
}
salida = agentes['hr'].invoke({'consulta': '¿Cuántos días de vacaciones tengo el primer año?'})
print(salida['respuesta'])
print('Fuentes:', salida['fuentes'])

El primer año tienes 15 días hábiles de vacaciones.
Fuentes: [{'archivo': 'vacaciones_y_licencias.md', 'seccion': 'Días de vacaciones por antigüedad'}, {'archivo': 'vacaciones_y_licencias.md', 'seccion': 'Vacaciones no tomadas y fraccionamiento'}]


## 4. Orquestador y enrutamiento

El orquestador devuelve una clasificación estructurada. `RunnableBranch` usa ese departamento para elegir el agente; lo que no es de nadie va a `fuera_de_dominio`.

In [5]:
orquestador = crear_orquestador(crear_llm(config))
for consulta in ['Me bloquearon la cuenta', '¿Cuánto son los viáticos nacionales?', 'Escríbeme un poema']:
    print(consulta, '->', orquestador.invoke({'consulta': consulta}))

sistema = crear_sistema(config, orquestador=orquestador, agentes=agentes)

Me bloquearon la cuenta -> {'departamento': 'tech', 'confianza': 0.99, 'razon': 'El bloqueo de una cuenta es un problema de acceso y soporte de TI.', 'error': False}


¿Cuánto son los viáticos nacionales? -> {'departamento': 'finance', 'confianza': 0.99, 'razon': 'Consulta el monto de los viáticos, un tema de finanzas.', 'error': False}


Escríbeme un poema -> {'departamento': 'fuera_de_dominio', 'confianza': 1.0, 'razon': 'Solicita una tarea creativa general, ajena al soporte interno de NubeGestor.', 'error': False}


## 5. Pruebas y ejemplos

Primero tres consultas sueltas (sin evaluador) y después las 15 test queries de `test_queries.json` para medir el acierto de enrutamiento.

In [6]:
for consulta in ['¿Cómo me reintegran el internet del home office?',
                 'Empiezo el lunes, ¿cuándo me entregan la laptop?',
                 '¿Qué tiempo hará mañana?']:
    salida = responder(consulta, config, sistema=sistema, con_evaluador=False)
    print(f"[{salida['departamento']}] {consulta}\n  {salida['respuesta']}\n")

[finance] ¿Cómo me reintegran el internet del home office?
  Carga la factura de internet en Mi NubeGestor > Gastos > Nuevo gasto, con la categoría home office, antes del día 5 del mes siguiente. Se paga con la nómina de ese mes y el subsidio de home office es de hasta USD 40 al mes; si tus facturas suman menos, se reintegra ese monto.



[tech] Empiezo el lunes, ¿cuándo me entregan la laptop?
  Tech entrega la laptop configurada el primer día de trabajo, es decir, el lunes. Si trabajas 100% en remoto, te la envían por mensajería el día hábil anterior.



[fuera_de_dominio] ¿Qué tiempo hará mañana?
  Solo puedo ayudarte con consultas de Recursos Humanos, Soporte de TI o Finanzas de NubeGestor. ¿Puedes reformular tu pregunta sobre alguno de esos temas?



In [7]:
metricas = medir_test_queries(config, 'test_queries.json',
                              'outputs/resultados_notebook.json', con_evaluador=False)

[indexación] hr: 86 chunks
[indexación] tech: 74 chunks
[indexación] finance: 70 chunks


[1] esperado=hr obtenido=hr error=None


[2] esperado=hr obtenido=hr error=None


[3] esperado=hr obtenido=hr error=None


[4] esperado=tech obtenido=tech error=None


...

[11] esperado=tech obtenido=tech error=None


[12] esperado=finance obtenido=finance error=None


[13] esperado=fuera_de_dominio obtenido=fuera_de_dominio error=None


[14] esperado=fuera_de_dominio obtenido=fuera_de_dominio error=None


[15] esperado=tech obtenido=tech error=None

Clasificaciones válidas: 15/15 (cobertura 100.0%)
Acierto de clasificación: 15/15 (100.0%)
  hr                3/3 válidas (3 consultas)
  tech              5/5 válidas (5 consultas)
  finance           5/5 válidas (5 consultas)
  fuera_de_dominio  2/2 válidas (2 consultas)
Errores técnicos por etapa: ninguno
Respuestas completadas: 15/15


## 6. Integración con Langfuse

`responder()` abre un span en Langfuse y pasa el `CallbackHandler` a LangChain: cada consulta queda como **una traza** con la clasificación, los chunks recuperados y la generación. Con el `trace_id` se abre la traza en la interfaz de Langfuse para depurar.

### Agente Evaluador (bonus)

Un LLM-juez puntúa relevance, completeness, accuracy y overall (1 a 10). Los cuatro puntajes se registran en la misma traza con la Score API de Langfuse.

In [8]:
salida = responder('Hice un curso de capacitación, ¿cómo presento la factura?', config, sistema=sistema)
print('Departamento:', salida['departamento'])
print('Respuesta:', salida['respuesta'])
print('Trace ID:', salida['trace_id'])
print('Evaluación:', salida['evaluacion'])

Departamento: finance
Respuesta: Pide al proveedor una factura a nombre de NubeGestor S.A.S., con el número de identificación tributaria que aparece en Mi NubeGestor > Finanzas > Datos de facturación. Cárgala en Mi NubeGestor > Gastos con la categoría «capacitación» y adjunta la aprobación de HR recibida por correo; sin esa aprobación, Finanzas no puede procesar el pago. Si la cargas antes del día 5, se paga con la nómina del mes.
Trace ID: d08b6243a69d8ece657c87a114d776b7
Evaluación: {'relevance': 10, 'completeness': 9, 'accuracy': 10, 'overall': 9, 'razon': 'La respuesta describe correctamente los pasos, la aprobación requerida y el plazo para el pago con nómina, todo respaldado por el contexto. No menciona la alternativa de pago directo al proveedor, aunque no era necesario para responder el caso general.'}
